In [ ]:
# Last amended: 23rd Nov, 2026
# My folder:
# GitHUB repo: llamaindex/revised14042026/14.graph_rag
#


# 🎓 Lab Tutorial: Understanding & Querying GraphRAG with LlamaIndex and Ollama

Welcome to this hands-on lab on **GraphRAG (Graph Retrieval-Augmented Generation)**!

---

## 📌 What You Will Learn Today:
1. **Why standard Vector RAG fails** on multi-hop relationships and cross-document connections.
2. **How Knowledge Graphs represent information** using `(Subject) ➔ [Predicate] ➔ (Object)` triples.
3. **How LlamaIndex extracts graphs** automatically using local open-weight LLMs via **Ollama**.
4. **How to visualize and interact with the graph**.
5. **How to query the graph** using hybrid graph traversal + vector retrieval.

## 1. Setup & Environment Verification
First, let's import the necessary modules and check our connection to the local **Ollama** server.

In [ ]:
# %pip install pyvis

In [1]:
%%time

# 1.0
# CRITICAL FOR JUPYTER NOTEBOOKS:
# Jupyter already runs inside an active asyncio event loop.
# nest_asyncio patches the loop so LlamaIndex can run its extraction calls seamlessly.
import nest_asyncio
nest_asyncio.apply()

CPU times: user 235 μs, sys: 45 μs, total: 280 μs
Wall time: 283 μs


In [2]:
# 1.1

import os
import sys
from pathlib import Path



In [ ]:
# 1.2 Base paths
PROJECT_ROOT = Path.cwd()
DATA_DIR = PROJECT_ROOT / "data"

# 1.3 Ensure output directory exists
# parents=True: Creates any missing parent directories 
#                along the path if they don't already exist.
# exist_ok=True: Prevents Python from raising a FileExistsError 
#                if the data folder is already there.
DATA_DIR.mkdir(parents=True, exist_ok=True)

## 2. Inspecting the Sample Dataset
Our dataset revolves around a complex corporate and research technology scenario involving four interconnected documents:
- `aurora_technologies.txt`: Quantum computing hardware firm and key founders.
- `helios_capital.txt`: Deep tech venture firm orchestrating cross-company investments.
- `synapse_dynamics.txt`: Advanced robotics company developing humanoid robots.
- `project_nexus_investigation.txt`: Investigative report revealing the covert integration of quantum chips into autonomous robotics.

Let's preview these documents:

In [3]:
# 2.0

# 2.1 Find all .txt files in DATA_DIR, sort them alphabetically,
#       and loop through each one
for doc_file in sorted(DATA_DIR.glob('*.txt')):
    # 2.1.1 Print the file's name (excluding the full directory path)
    print(f"📄 File: {doc_file.name}")
    # 2.1.2 Open the current file in read mode ('r') and 
    #       ensure it closes automatically afterward
    with open(doc_file, 'r') as f:
        # 2.1.3 Read the first 250 characters and replace line breaks
        #        with spaces for a clean single-line preview
        preview = f.read(250).replace('\n', ' ')
        # 2.1.4 Print the formatted preview followed 
        #       by an extra blank line for spacing
        print(f"   Preview: {preview}...\n")


📄 File: aurora_technologies.txt
   Preview: # Company Overview: Aurora Technologies Inc.  Aurora Technologies is a pioneering quantum computing hardware company founded in 2021 by Dr. Elena Vance and Marcus Thorne. Headquartered in Seattle, Washington, Aurora specializes in room-temperature su...

📄 File: helios_capital.txt
   Preview: # Investment Dossier: Helios Capital Management  Helios Capital is a venture capital and private equity firm based in New York City, specializing in deep tech, quantum infrastructure, and autonomous systems. Helios is managing partner for the $1.2B "...

📄 File: project_nexus_investigation.txt
   Preview: # Investigative Report: The Secret Architecture of Project Nexus Source: Tech Sentinel Investigative Desk Classification: Public Domain Investigation Summary  In late 2024, confidential whistleblower memos revealed the existence of "Project Nexus", a...

📄 File: synapse_dynamics.txt
   Preview: # Corporate Profile: Synapse Dynamics  Synapse Dynamics is 

## 3. Configuring LlamaIndex with Local Ollama
We configure LlamaIndex to use:
- **LLM**: `llama3.2` (or `llama3`, `mistral`, `qwen2.5`)
- **Embeddings**: `nomic-embed-text`

In [4]:
%%time

# 3.0
from llama_index.core import Settings, SimpleDirectoryReader
from llama_index.llms.ollama import Ollama
from llama_index.embeddings.ollama import OllamaEmbedding


# 3.1
llm = Ollama(model="qwen2.5:latest",
             request_timeout=800.0,
             temperature = 0.9
            )

# 3.2 Global Embedding Model
embed_model = OllamaEmbedding(model_name="nomic-embed-text:latest")

# 3.3
Settings.llm = llm
Settings.embed_model = embed_model
Settings.chunk_size = 512



Loaded 4 document sections.


In [ ]:
%%time

# 4.0 Read files
reader = SimpleDirectoryReader(input_dir=str(DATA_DIR))
documents = reader.load_data()
print(f"Loaded {len(documents)} document sections.")

## 4. Building the PropertyGraphIndex (GraphRAG)
The core step of GraphRAG: the LLM scans each chunk and extracts **Entities** (people, organizations, technologies) and **Relations** between them into a persistent graph store.

This code sets up a knowledge graph extractor in LlamaIndex that uses a Large Language Model (LLM) to pull structured facts from text in the form of (Entity) -> [Relation] -> (Entity) triplets.

Commented Code

python
### 4.1 Extract knowledge graph (kg) 
#####     entity-relation-entity paths

#### Initialize the extractor that uses an LLM to identify relationships in text
```bash
kg_extractor = SimpleLLMPathExtractor(
    # The Large Language Model instance (e.g., OpenAI, Ollama) used to read the text and extract facts
    llm=llm,
    
    # Caps the extraction at 10 relationship triplets per text chunk to prevent excessive API usage or noise
    max_paths_per_chunk=10,
    
    # Processes 1 chunk at a time sequentially (useful for avoiding API rate limits or local GPU memory spikes)
    num_workers=1
)
```
### How It Works

When you pass documents into a LlamaIndex pipeline (like PropertyGraphIndex), this component scans chunks of text and turns unstructured sentences into structured graph connections.      
• Input Text: "Elon Musk founded SpaceX in 2002."       
• Extracted Path: (Elon Musk) —[FOUNDED]→ (SpaceX)           

Parameter Breakdown              

>llm: The brain doing the extraction. It reads your text chunks and follows an internal prompt to output structured triplets.      
> max_paths_per_chunk: A quality and cost control setting. If a single paragraph has 30 potential facts, the extractor will only keep up to 10 of the most prominent ones.      
> num_workers: Controls concurrency. Setting this to 1 disables parallel processing, meaning chunks are processed one after another. Increasing this (e.g., to 4 or 8) speeds up indexing significantly if your LLM provider allows high rate limits.     


In [5]:
%%time

# 5.0 Required for Jupyter: allow nested event loops
from llama_index.core import PropertyGraphIndex
from llama_index.core.indices.property_graph import SimpleLLMPathExtractor

# 5.1 Extract knowledge graph (kg) 
#     entity-relation-entity paths

# 5.1.1 Initialize the extractor that uses 
#       an LLM to identify relationships in text
kg_extractor = SimpleLLMPathExtractor(
                                        # 5.1.2 The Large Language Model instance 
                                        #       used to read the text and extract facts
                                        llm=llm,
                                        
                                        # 5.1.3 Caps the extraction at 10 relationship triplets
                                        #        per text chunk to prevent excessive API usage or noise
                                        max_paths_per_chunk=10,
                                        
                                        # 5.1.4 Processes 1 chunk at a time sequentially 
                                        #       (useful for avoiding API rate limits or 
                                        #       local GPU memory spikes)
                                        num_workers=1    # Try to make it 2
                                    )

Constructing Property Graph Index via Ollama...


Applying transformations:   0%|          | 0/1 [00:00<?, ?it/s]

Applying transformations:   0%|          | 0/1 [00:00<?, ?it/s]

2026-11-26 15:33:53,463 - INFO - HTTP Request: POST http://localhost:11434/api/show "HTTP/1.1 200 OK"
2026-11-26 15:34:14,614 - INFO - HTTP Request: POST http://localhost:11434/api/chat "HTTP/1.1 200 OK"
2026-11-26 15:34:45,668 - INFO - HTTP Request: POST http://localhost:11434/api/chat "HTTP/1.1 200 OK"
2026-11-26 15:35:02,623 - INFO - HTTP Request: POST http://localhost:11434/api/chat "HTTP/1.1 200 OK"
2026-11-26 15:35:23,972 - INFO - HTTP Request: POST http://localhost:11434/api/chat "HTTP/1.1 200 OK"


Generating embeddings:   0%|          | 0/4 [00:00<?, ?it/s]

2026-11-26 15:35:25,262 - INFO - HTTP Request: POST http://localhost:11434/api/embed "HTTP/1.1 200 OK"


Generating embeddings:   0%|          | 0/168 [00:00<?, ?it/s]

2026-11-26 15:35:27,138 - INFO - HTTP Request: POST http://localhost:11434/api/embed "HTTP/1.1 200 OK"
2026-11-26 15:35:29,007 - INFO - HTTP Request: POST http://localhost:11434/api/embed "HTTP/1.1 200 OK"
2026-11-26 15:35:30,874 - INFO - HTTP Request: POST http://localhost:11434/api/embed "HTTP/1.1 200 OK"
2026-11-26 15:35:32,747 - INFO - HTTP Request: POST http://localhost:11434/api/embed "HTTP/1.1 200 OK"
2026-11-26 15:35:34,617 - INFO - HTTP Request: POST http://localhost:11434/api/embed "HTTP/1.1 200 OK"
2026-11-26 15:35:36,485 - INFO - HTTP Request: POST http://localhost:11434/api/embed "HTTP/1.1 200 OK"
2026-11-26 15:35:38,356 - INFO - HTTP Request: POST http://localhost:11434/api/embed "HTTP/1.1 200 OK"
2026-11-26 15:35:40,231 - INFO - HTTP Request: POST http://localhost:11434/api/embed "HTTP/1.1 200 OK"
2026-11-26 15:35:42,102 - INFO - HTTP Request: POST http://localhost:11434/api/embed "HTTP/1.1 200 OK"
2026-11-26 15:35:43,975 - INFO - HTTP Request: POST http://localhost:1143

Graph index built successfully!


In [ ]:
%%time

# 6.0
print("Constructing Property Graph Index via Ollama...")
# Note: use_async=False prevents "RuntimeError: asyncio.run() 
#    cannot be called from a running event loop" in Jupyter

# 6.1
prop_index = PropertyGraphIndex.from_documents(
                                                documents,
                                                kg_extractors=[kg_extractor],
                                                use_async=False,
                                                show_progress=True
                                                )
# 6.1.1
print("Graph index built successfully!")

In [8]:
%%time

# 7.0
from IPython.display import IFrame

# 7.1. Export the graph to a standalone HTML file using PyVis
prop_index.property_graph_store.save_networkx_graph(name="./graph.html")

# 7.2. Display the HTML file directly inside your notebook cell
IFrame(src="./graph.html", width="100%", height="800px")


## 5. Querying: Vector RAG vs. GraphRAG Comparison
Now let's ask a multi-hop question that requires bridging information across **3 distinct documents**:

> *"How is Dr. Elena Vance connected to Synapse Dynamics and Project Nexus through Helios Capital?"*

In [9]:
%%time

# 8.0
query_text = "How is Dr. Elena Vance connected to Synapse Dynamics and Project Nexus through Helios Capital?"

# 8.1. GraphRAG Query Engine
graph_engine = prop_index.as_query_engine(
                                            include_text=True, 
                                            similarity_top_k=3
                                          )

# 8.2
response_graph = graph_engine.query(query_text)

print("=== 💡 GRAPHRAG RESPONSE ===")
print(response_graph.response)

2026-11-26 15:37:40,705 - INFO - HTTP Request: POST http://localhost:11434/api/chat "HTTP/1.1 200 OK"
2026-11-26 15:37:40,726 - INFO - HTTP Request: POST http://localhost:11434/api/embed "HTTP/1.1 200 OK"
2026-11-26 15:37:54,578 - INFO - HTTP Request: POST http://localhost:11434/api/chat "HTTP/1.1 200 OK"


=== 💡 GRAPHRAG RESPONSE ===
Dr. Elena Vance is connected to Synapse Dynamics through her appointment as the Chief Scientific Advisor, a role she was appointed to in March 2024. Her connection to Project Nexus is indirect, as she is involved in Synapse Dynamics' work, including the company's efforts on Project Nexus, though the specific nature of her involvement is not detailed in the provided context. Helios Capital is one of Synapse Dynamics' major equity investors, contributing $60M in Series C funding, but the direct connection between Dr. Elena Vance and Helios Capital in the context provided is not explicitly stated.


## 6. Hands-on Student Exercises

1. **Trace a New Pathway**: Add a new document to `data/` about a government regulatory hearing (`senate_hearing.txt`) and rebuild the graph.
2. **Entity Types**: Modify `SimpleLLMPathExtractor` to specify exact schemas: `schema={'PERSON': ['CO_FOUNDED', 'ADVISED', 'COLLABORATED_WITH']}`.
3. **Compare Retrieval Top-K**: Test how increasing `similarity_top_k` in Vector RAG still struggles with relational queries compared to Graph traversal.

In [ ]:
###############